MÔ HÌNH PEAS CHO 8-PUZZLE (MODEL - BASED)
- P (Performance): Đưa các ô số về đúng vị trí đích (1-8), tổng số bước di chuyển là ít nhất, thời gian giải nhanh.
- E (Environment): Ma trận 3x3 chứa các số từ 1 đến 8 và một ô trống (ký hiệu là 0); khung bàn cờ, các ô số.
- A (Actuators): Các hàm di chuyển ô trống (UP, DOWN, LEFT, RIGHT); cơ chế tráo đổi vị trí các ô số.
- S (Sensors): Hàm đọc dữ liệu ma trận hiện tại, xác định vị trí tọa độ (x, y) của ô trống.

PERCEPT (NHẬN THỨC)
- Ma trận 3x3, giá trị từ 1-8 là các ô số, giá trị 0 là vị trí ô trống.
- Trạng thái đích (Goal):
                1 2 3
                4 5 6
                7 8 0

RULES (TẬP LUẬT)
- Hàm xác định các hướng có thể đi dựa trên tọa độ $(x, y) của ô trống:
    possible_move (x, y):
        if x > 0: move.append(UP) (Ô trống không ở hàng trên cùng).
        if x < 2: move.append(DOWN) (Ô trống không ở hàng dưới cùng).
        if y > 0: move.append(LEFT) (Ô trống không ở cột ngoài cùng bên trái).
        if y < 2: move.append(RIGHT) (Ô trống không ở cột ngoài cùng bên phải).
        -> return move
    Action: action = RAND(move) (Chọn ngẫu nhiên một hướng trong danh sách để di chuyển).
- Xác định hướng đi ngẫu nhiên (RAND) dựa trên vị trí cụ thể của ô trống trên ma trận 3 x 3:
    Nếu ở 4 góc:
        if state = (0, 0) (Góc trên trái) then action = RAND(DOWN, RIGHT).
        if state = (0, 2) (Góc trên phải) then action = RAND(DOWN, LEFT).
        if state = (2, 0) (Góc dưới trái) then action = RAND(UP, RIGHT).
        if state = (2, 2) (Góc dưới phải) then action = RAND(UP, LEFT).
    Nếu ở các cạnh (không phải góc):
        if state = (0, 1) (Cạnh trên) then action = RAND(DOWN, LEFT, RIGHT).
        if state = (2, 1) (Cạnh dưới) then action = RAND(UP, LEFT, RIGHT).
        if state = (1, 0) (Cạnh trái) then action = RAND(UP, DOWN, RIGHT).
        if state = (1, 2) (Cạnh phải) then action = RAND(UP, DOWN, LEFT).
    Nếu ở giữa:
        if state = (1, 1) then action = RAND(UP, DOWN, LEFT, RIGHT).

In [1]:
class ModelBasedAgent:
    def __init__(self, start, target):
        self.state = start
        self.target = target
        self.history = set()
        self.history.add(start)
    def get_valid_moves(self, state):
        moves = []
        r, c = next((r, c) for r, row in enumerate(state) for c, val in enumerate(row) if val == 0)
        directions = [(-1, 0, "Lên"), (1, 0, "Xuống"), (0, -1, "Trái"), (0, 1, "Phải")]
        for dr, dc, label in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_board = [list(row) for row in state]
                new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
                new_state = tuple(tuple(row) for row in new_board)
                if new_state not in self.history:
                    moves.append((new_state, label))
        return moves
    def run(self):
        current = self.state
        step = 0
        print("Bắt đầu giải ma trận:")
        self.print_board(current)
        while current != self.target:
            possible_moves = self.get_valid_moves(current)
            if not possible_moves:
                print(">>> Agent đã thử hết mọi nước đi khả thi nhưng không thể tới đích.")
                print(">>> Nguyên nhân: Ma trận này bị nghịch thế lẻ, không thể về 7 8 0.")
                break
            def h(s):
                return sum(1 for i in range(3) for j in range(3) if s[i][j] != self.target[i][j])
            current, move_label = min(possible_moves, key=lambda x: h(x[0]))
            self.history.add(current)
            step += 1
            print(f"Bước {step}: Di chuyển ô trống {move_label}")
            self.print_board(current)
    def print_board(self, s):
        for row in s:
            print(" ".join(str(x) if x != 0 else "_" for x in row))
        print("-" * 10)
start_state = ((1, 2, 3), 
               (4, 5, 6), 
               (8, 0, 7))
target_state = ((1, 2, 3), 
                (4, 5, 6), 
                (7, 8, 0))

agent = ModelBasedAgent(start_state, target_state)
agent.run()

Bắt đầu giải ma trận:
1 2 3
4 5 6
8 _ 7
----------
Bước 1: Di chuyển ô trống Trái
1 2 3
4 5 6
_ 8 7
----------
Bước 2: Di chuyển ô trống Lên
1 2 3
_ 5 6
4 8 7
----------
Bước 3: Di chuyển ô trống Lên
_ 2 3
1 5 6
4 8 7
----------
Bước 4: Di chuyển ô trống Phải
2 _ 3
1 5 6
4 8 7
----------
Bước 5: Di chuyển ô trống Xuống
2 5 3
1 _ 6
4 8 7
----------
Bước 6: Di chuyển ô trống Trái
2 5 3
_ 1 6
4 8 7
----------
Bước 7: Di chuyển ô trống Xuống
2 5 3
4 1 6
_ 8 7
----------
Bước 8: Di chuyển ô trống Phải
2 5 3
4 1 6
8 _ 7
----------
Bước 9: Di chuyển ô trống Phải
2 5 3
4 1 6
8 7 _
----------
Bước 10: Di chuyển ô trống Lên
2 5 3
4 1 _
8 7 6
----------
Bước 11: Di chuyển ô trống Trái
2 5 3
4 _ 1
8 7 6
----------
Bước 12: Di chuyển ô trống Lên
2 _ 3
4 5 1
8 7 6
----------
Bước 13: Di chuyển ô trống Trái
_ 2 3
4 5 1
8 7 6
----------
Bước 14: Di chuyển ô trống Xuống
4 2 3
_ 5 1
8 7 6
----------
Bước 15: Di chuyển ô trống Xuống
4 2 3
8 5 1
_ 7 6
----------
Bước 16: Di chuyển ô trống Phải
4 2 3
8 5 1